In [0]:
%run ../04_Ukey_Match/ukey_match_regular_common_business

In [0]:
def get_pre_data(task_id, consumer_select_condition_str, t_merge_exclude_consumer_config, t_merge_exclude_phone_config, t_merge_exclude_media_config, t_merge_exclude_address_config):

    # 1.筛选consumer
    filter_consumer_df = spark.table(f"{get_env_config('golden_consumer_master_database')}.t_master_consumer") \
        .where(consumer_select_condition_str)

    # filter_consumer_df.cache()
    print(f"filter_consumer_df: {filter_consumer_df.count()}")

    acsline_consumer = (
        filter_consumer_df
        .join(t_merge_exclude_consumer_config,
            (F.col("scon_srcs_code") == F.col("exclude_srcs_code")) &
            (F.col("scon_mrkt_code") == F.col("exclude_mrkt")),
            "inner")
        .withColumn("is_master_recode", F.lit(False))
        .withColumn("master_recode_create_time", F.lit(None).cast(TimestampType()))
    )


    t_master_consumer = (
        filter_consumer_df
        .join(t_merge_exclude_consumer_config,
            (F.col("scon_srcs_code") == F.col("exclude_srcs_code")) &
            (F.col("scon_mrkt_code") == F.col("exclude_mrkt")),
            "left_anti")
        .withColumn("is_master_recode", F.lit(False))
        .withColumn("master_recode_create_time", F.lit(None).cast(TimestampType()))
    )

    # 2. phone
    t_master_phone_vld = (
        spark.table(f"{get_env_config('golden_consumer_master_database')}.t_master_phone")
        .filter(F.col("scph_quality_code") == "vld")
        .filter(F.coalesce(F.col("scph_phonenumber"), F.lit("")) != "")
        .join(t_merge_exclude_phone_config,
            (F.col("scph_phonenumber") == F.col("phone")) &
            (F.col("scph_mrkt_code") == F.col("exclude_mrkt")),
            "left_anti")
    )

    # 3. email
    t_master_emedia_vld = (
        spark.table(f"{get_env_config('golden_consumer_master_database')}.t_master_emedia")
        .filter(
            (F.col("scme_quality_code") == "vld") |
            (F.col("scme_mrkt_code").isin(MARKETS_ENABLE_LINE_MEDIA) & (F.col("scme_emdt_code") == "scllneprs"))
        )
        .join(t_merge_exclude_media_config,
            (F.col("scme_address") == F.col("email")) &
            (F.col("scme_mrkt_code") == F.col("exclude_mrkt")),
            "left_anti")
        .withColumn("scme_address", F.lower(F.col("scme_address")))
    )


    # 4. address
    t_master_address = (
        spark.table(f"{get_env_config('golden_consumer_master_database')}.t_master_address")
        .filter(F.trim(F.coalesce(F.col("scad_address1"), F.lit(""))) != "")
        .join(t_merge_exclude_address_config,
            (F.col("scad_address1") == F.col("addr")) &
            (F.col("scad_mrkt_code") == F.col("exclude_mrkt")),
            "left_anti")
    )


    # 5. 
    t_master_consumergroup = (
        spark.table(f"{get_env_config('golden_consumer_master_database')}.t_master_consumer_group")
    )


    output_df = (
        t_master_consumer.alias("a")
        .join(t_master_phone_vld.alias("b"),
            (F.col("b.scph_scon_id") == F.col("a.scon_id")) &
            (F.col("b.scph_mrkt_code") == F.col("a.scon_mrkt_code")),
            "left")
        .join(t_master_emedia_vld.alias("c"),
            (F.col("c.scme_scon_id") == F.col("a.scon_id")) &
            (F.col("c.scme_mrkt_code") == F.col("a.scon_mrkt_code")),
            "left")
        .join(t_master_address.alias("d"),
            (F.col("d.scad_scon_id") == F.col("a.scon_id")) &
            (F.col("d.scad_mrkt_code") == F.col("a.scon_mrkt_code")) &
            (~F.col("a.scon_mrkt_code").isin(MARKETS_DISABLE_ADDRESS_PATH4)),
            "left")
        .join(t_master_consumergroup.alias("g"),
            (F.col("g.scgr_scon_id") == F.col("a.scon_id")) &
            (F.col("g.scgr_mrkt_code") == F.col("a.scon_mrkt_code")),
            "left")
        .select(
            F.col("scon_id"),
            F.lit(None).cast("string").alias("consumermdmkey"),

            F.col("scon_srcc_id"),
            F.col("scon_srcs_code"),
            F.col("scon_sourcetimestamp"),
            F.col("scon_mrkt_code"),
            F.col("scon_brnd_code"),
            F.col("scon_consumerid"),

            F.col("scon_englishfirstname"),
            F.col("scon_englishmiddlename"),
            F.col("scon_englishlastname"),
            F.col("scon_englishfullname"),

            F.col("scon_localfirstname"),
            F.col("scon_localmiddlename"),
            F.col("scon_locallastname"),
            F.col("scon_localfullname"),

            F.col("scon_localfirstname2"),
            F.col("scon_localmiddlename2"),
            F.col("scon_locallastname2"),
            F.col("scon_localfullname2"),

            F.coalesce(F.col("c.scme_address"), F.lit("")).alias("scme_address"),
            F.coalesce(F.col("b.scph_phonenumber"), F.lit("")).alias("scph_phonenumber"),
            F.coalesce(F.trim(F.col("d.scad_address1")), F.lit("")).alias("scad_address1"),
            F.coalesce(F.trim(F.col("d.scad_address2")), F.lit("")).alias("scad_address2"),
            F.coalesce(F.trim(F.col("d.scad_address3")), F.lit("")).alias("scad_address3"),
            F.coalesce(F.trim(F.col("d.scad_city_localdesc")), F.lit("")).alias("scad_city_localdesc"),
            F.coalesce(F.trim(F.col("d.scad_postalcode")), F.lit("")).alias("scad_postalcode"),

            F.col("a.is_master_recode"),
            F.col("a.master_recode_create_time"),
            F.col("a.batch_id"),

            # 差异 2: OtherBlockingKey（PHL→Normal, KOR Brand43→DrJart, Brand99+empl→EmpBrand99）
            build_other_blocking_key(
                "a.scon_mrkt_code",
                "a.scon_brnd_code",
                "g.scgr_consumer_grp"
            ).alias("OtherBlockingKey"),
        ).distinct()
    )


    return acsline_consumer, output_df



In [0]:
def ukey_match_acsline_process(task_id, acsline_consumer, regular_final_df):
    """
    为 split scope 内的 ACS/LineBind consumer 生成 ukey。
    三级优先级：
      P1: consumermdmkey 匹配 master 表不同 source → 复用 master ukey
      P2: 同市场+品牌+consumerid 匹配 master 表 → 复用 master ukey
      P3: 同市场+品牌+consumerid 匹配 split scope regular 结果 → 复用 regular 新 ukey
      Fallback: 保留原 consumermdmkey
    最后通过 regular 结果中的 recode 信息修正为最新 ukey。
    """

    # 1. master consumer lookup（全局，用于 P1 & P2）
    t_master_consumer = (
        spark.table(f"{get_env_config('golden_consumer_master_database')}.t_master_consumer")
        .select(
            F.col("scon_mrkt_code"),
            F.col("scon_brnd_code"),
            F.col("scon_srcs_code"),
            F.col("scon_consumerid"),
            F.col("consumermdmkey")
        )
    )

    # 2. split scope regular 结果 → Priority 3 lookup
    regular_clean_df = (
        regular_final_df
        .select(
            F.col("mrkt_code"),
            F.col("brnd_code"),
            F.col("consumer_id"),
            F.col("new_consumermdmkey")
        )
        .distinct()
    )

    # 3. 三级优先级生成 acsline ukey
    acsline_ukey_df = (
        acsline_consumer.alias("ac")
        # P1: 同 market + 同 consumermdmkey + 不同 source → 找到其他 source 的相同 ukey
        .join(
            t_master_consumer.alias("mc_tab1"),
            (F.col("ac.scon_mrkt_code") == F.col("mc_tab1.scon_mrkt_code")) &
            (F.col("ac.scon_universalkey") == F.col("mc_tab1.consumermdmkey")) &
            (F.col("ac.scon_srcs_code") != F.col("mc_tab1.scon_srcs_code")),
            "left"
        )
        # P2: 同 market + 同 brand + 同 consumerid → split scope regular 结果
        .join(
            regular_clean_df.alias("rc_tab"),
            (F.col("ac.scon_mrkt_code") == F.col("rc_tab.mrkt_code")) &
            (F.col("ac.scon_brnd_code") == F.col("rc_tab.brnd_code")) &
            (F.col("ac.SCON_MASTERCONSUMERID") == F.col("rc_tab.consumer_id")),
            "left"
        )
        # P3: 同 market + 同 brand + 同 consumerid → 找到 master 中同品牌同ID的记录
        .join(
            t_master_consumer.alias("mc_tab2"),
            (F.col("ac.scon_mrkt_code") == F.col("mc_tab2.scon_mrkt_code")) &
            (F.col("ac.scon_brnd_code") == F.col("mc_tab2.scon_brnd_code")) &
            (F.col("ac.SCON_MASTERCONSUMERID") == F.col("mc_tab2.scon_consumerid")),
            "left"
        )
        .withColumn("new_consumermdmkey",
            F.when(
                F.coalesce(F.col("mc_tab1.consumermdmkey"), F.lit("")) != "",
                F.col("mc_tab1.consumermdmkey")
            ).when(
                F.coalesce(F.col("rc_tab.new_consumermdmkey"), F.lit("")) != "",
                F.col("rc_tab.new_consumermdmkey")
            ).when(
                F.coalesce(F.col("mc_tab2.consumermdmkey"), F.lit("")) != "",
                F.col("mc_tab2.consumermdmkey")
            ).otherwise(
                F.col("ac.consumermdmkey")
            )
        )
        .select(
            F.expr("uuid()").alias("matc_id"),
            F.col("ac.scon_srcc_id").alias("srcc_id"),
            F.col("ac.scon_mrkt_code").alias("mrkt_code"),
            F.col("ac.scon_brnd_code").alias("brnd_code"),
            F.col("ac.scon_srcs_code").alias("source_code"),
            F.col("ac.scon_consumerid").alias("consumer_id"),
            F.col("ac.scon_sourcetimestamp").alias("source_timestamp"),
            F.col("new_consumermdmkey"),
            F.col("ac.scon_id").alias("master_scon_id"),
            F.col("ac.consumermdmkey").alias("master_consumermdmkey"),
            F.col("ac.master_recode_create_time"),
            F.col("ac.is_master_recode"),
            F.lit(None).alias("gid"),
            F.lit(None).alias("grp_size"),
            F.col("ac.batch_id"),
            F.lit(task_id).alias("task_id"),
            F.lit(MATCH_TYPE_ACSLINE_SPLIT_STR).alias("match_type"),
            F.current_timestamp().alias("creation_dt")
        )
        .withColumn("row_num", F.row_number().over(Window.partitionBy("mrkt_code", "srcc_id").orderBy(F.col("new_consumermdmkey"))))
        .filter(F.col("row_num") == 1)
        .drop("row_num")
    )

    # 5. union regular + acsline
    return regular_final_df.unionByName(acsline_ukey_df)


In [0]:
def ukey_match_regular_process(task_id, consumer_select_condition_str):
    t_merge_exclude_phone_config, t_merge_exclude_media_config, t_merge_exclude_address_config, t_merge_exclude_consumer_config, exclude_phones, exclude_emails = get_exclude_dfs()

    # 1.1 select data
    acsline_consumer, output_df = get_pre_data(task_id, consumer_select_condition_str, t_merge_exclude_consumer_config, t_merge_exclude_phone_config, t_merge_exclude_media_config, t_merge_exclude_address_config)

    # 1.3 batch union master, generate match key
    merged_df = generate_match_key(output_df)

    # 2.1 generate vertices, edges
    df_with_id = merged_df.withColumn("id", F.concat_ws("_", F.col("scon_mrkt_code"), F.col("scon_srcc_id")))
    vertices, edges = generate_vertices_and_edge(df_with_id)

    df_with_id.cache()
    edges.cache()

    save_to_target_table(
        edges.withColumn("task_id", F.lit(task_id)).withColumn("creation_dt", F.current_timestamp()),
        f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_ukey_edge",
        f"task_id = '{task_id}'"
    )
    
    # 2.2 generate gid
    final_df = generate_gid(vertices, edges, df_with_id) \
        .withColumn("match_type", F.lit(MATCH_TYPE_REGULAR_SPLIT_STR)) \

    # 3. new Ukey generate
    final_with_newUkey_df = generate_new_ukey(final_df)

    # 4. acs&line ukey generate（split scope 内重新绑定）
    newUkey_df = ukey_match_acsline_process(task_id, acsline_consumer, final_with_newUkey_df)

    save_to_target_table(
        newUkey_df,
        f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_ukey_group",
        f"task_id = '{task_id}' "
    )

    df_with_id.unpersist()
    edges.unpersist()

    # 4. update ukey
    # update_ukey_to_master_table(task_id)
    

In [0]:
task_id = dbutils.widgets.get("task_id")
print(f"task_id: {task_id}")

consumer_select_condition_str = dbutils.widgets.get("consumer_select_condition_str")
print(f"consumer_select_condition_str: {consumer_select_condition_str}")
if consumer_select_condition_str is None or consumer_select_condition_str.strip() == "" :
    raise ValueError("consumer_select_condition_str错误:参数无效")

ukey_match_regular_process(task_id, consumer_select_condition_str)